# 01 — Análise Exploratória do Dataset EEG

Este notebook tem como objetivo realizar a análise inicial do dataset utilizado no projeto de TCC.

O dataset contém dados de eletroencefalograma (EEG) relacionados a movimentos das mãos, coletados de cinco participantes. Nesta etapa, o foco é compreender a estrutura dos dados, verificar a quantidade de amostras, identificar as classes, analisar a distribuição por usuário e validar a ausência de problemas básicos, como valores ausentes.

Esta análise é importante antes da implementação dos modelos de Inteligência Artificial, pois permite compreender melhor o comportamento do conjunto de dados e orientar as próximas etapas do projeto.

## 1. Configuração do ambiente

Nesta etapa são configurados os caminhos do projeto e importadas as bibliotecas necessárias. Como o notebook está localizado na pasta `notebooks`, é necessário adicionar a raiz do projeto ao caminho de execução do Python para permitir a importação dos módulos localizados em `src`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Raiz do projeto:", PROJECT_ROOT)

## 2. Importação do loader do dataset

O módulo `loader.py` foi criado para centralizar a leitura dos arquivos CSV do dataset. Ele carrega os dados dos cinco usuários e retorna um único DataFrame, adicionando uma coluna chamada `user` para identificar de qual participante cada amostra foi originada.

In [ ]:
from src.data.loader import load_all_users

df = load_all_users()

print("Dataset carregado com sucesso.")

## 3. Visualização inicial do dataset

A seguir, são exibidas as primeiras linhas do dataset e suas dimensões. A quantidade de linhas representa o número total de amostras, enquanto a quantidade de colunas representa as variáveis disponíveis, incluindo as features de EEG, o rótulo da classe e a identificação do usuário.

In [ ]:
print("Dimensões do dataset:", df.shape)

df.head()

### Interpretação inicial

O dataset possui milhares de amostras distribuídas entre cinco participantes. Cada linha representa uma amostra do conjunto de dados, enquanto as colunas representam as características extraídas dos sinais EEG, o rótulo da classe e o usuário correspondente.

A coluna `Values` representa a classe da amostra, enquanto a coluna `user` identifica o participante.

## 4. Identificação das colunas do dataset

Nesta etapa são separadas as colunas de entrada dos modelos, ou seja, as features utilizadas para classificação, removendo as colunas `Values` e `user`.

In [ ]:
feature_cols = [col for col in df.columns if col not in ["Values", "user"]]

print("Quantidade de features:", len(feature_cols))
print("Features encontradas:")
feature_cols

## 5. Identificação dos canais e bandas de frequência

As features do dataset seguem o padrão `POW.CANAL.BANDA`, indicando que os dados já foram previamente processados e representam informações espectrais dos sinais EEG.

Nesta etapa, são identificados os canais EEG e as bandas de frequência presentes no dataset.

In [ ]:
channels = sorted(set(col.split(".")[1] for col in feature_cols))
bands = sorted(set(col.split(".")[2] for col in feature_cols))

print("Canais EEG encontrados:")
print(channels)

print("\nBandas de frequência encontradas:")
print(bands)

### Interpretação

A identificação dos canais e bandas permite compreender a estrutura das características utilizadas no projeto. Como o dataset já possui features no domínio da frequência, não será necessário aplicar FFT ou PSD diretamente sobre sinais brutos nesta etapa inicial.

Assim, o projeto parte de características espectrais previamente extraídas, que serão utilizadas como entrada para os modelos de Inteligência Artificial.

## 6. Distribuição de amostras por usuário

Como o dataset possui cinco participantes, é importante verificar quantas amostras pertencem a cada um deles. Essa análise ajuda a identificar possíveis desequilíbrios entre usuários.

In [ ]:
user_counts = df["user"].value_counts()

print(user_counts)

In [ ]:
import matplotlib.pyplot as plt

user_counts.plot(kind="bar")

plt.title("Quantidade de amostras por usuário")
plt.xlabel("Usuário")
plt.ylabel("Quantidade de amostras")
plt.xticks(rotation=0)
plt.show()

### Interpretação

A distribuição por usuário permite verificar se todos os participantes possuem quantidades semelhantes de amostras. Caso existam grandes diferenças, isso pode influenciar o treinamento dos modelos, principalmente em estratégias de avaliação por participante.

## 7. Distribuição das classes

A coluna `Values` representa os rótulos das classes. Nesta etapa, é verificada a quantidade de amostras por classe, a fim de identificar se o dataset está balanceado.

In [ ]:
class_counts = df["Values"].value_counts().sort_index()
class_percent = df["Values"].value_counts(normalize=True).sort_index() * 100

print("Quantidade por classe:")
print(class_counts)

print("\nPercentual por classe:")
print(class_percent.round(2))

In [ ]:
class_counts.plot(kind="bar")

plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade de amostras")
plt.xticks(rotation=0)
plt.show()

### Interpretação

A análise da distribuição das classes permite avaliar se o problema de classificação apresenta desequilíbrio. Um dataset muito desbalanceado pode fazer com que os modelos tenham tendência a prever com maior frequência a classe dominante.

Neste caso, a distribuição será considerada nas próximas etapas, principalmente durante a divisão entre treino e teste e na escolha das métricas de avaliação.

## 8. Distribuição das classes por usuário

Além de verificar a distribuição geral das classes, é necessário analisar como as classes estão distribuídas dentro de cada usuário. Essa análise é importante porque um participante pode possuir mais amostras de uma classe do que de outra.

In [ ]:
class_by_user = pd.crosstab(df["user"], df["Values"])

class_by_user

In [ ]:
class_by_user_percent = pd.crosstab(
    df["user"], 
    df["Values"], 
    normalize="index"
) * 100

class_by_user_percent.round(2)

### Interpretação

A distribuição das classes por usuário auxilia na definição da estratégia de avaliação dos modelos. Como o dataset possui múltiplos participantes, uma possibilidade futura é avaliar se os modelos conseguem generalizar para usuários não vistos durante o treinamento.

## 9. Verificação de valores ausentes

Antes de treinar qualquer modelo, é necessário verificar se existem valores ausentes no dataset. A presença de valores nulos poderia exigir tratamento adicional, como remoção ou imputação.

In [ ]:
missing_values = df.isnull().sum().sum()

print("Quantidade total de valores ausentes:", missing_values)

### Interpretação

Caso o resultado seja zero, significa que o dataset não possui valores ausentes e pode seguir para as próximas etapas sem necessidade de tratamento de dados nulos.

## 10. Estatísticas descritivas das features

Nesta etapa são calculadas estatísticas básicas das features, como média, desvio padrão, valores mínimos e máximos. Isso permite observar a escala dos dados e possíveis diferenças entre as variáveis.

In [ ]:
df[feature_cols].describe()

### Interpretação

As estatísticas descritivas ajudam a verificar a escala das features. Caso existam diferenças muito grandes entre os valores das colunas, pode ser necessário aplicar normalização ou padronização antes do treinamento dos modelos.

## 11. Conclusões iniciais da análise exploratória

A análise exploratória permitiu compreender a estrutura inicial do dataset. Foram identificadas as dimensões gerais dos dados, a quantidade de participantes, a distribuição das classes, a organização das features e a ausência de valores ausentes.

Também foi observado que o dataset contém características espectrais previamente extraídas dos sinais EEG, organizadas por canais e bandas de frequência. Dessa forma, as próximas etapas do projeto devem se concentrar na preparação dos dados para treinamento, na divisão adequada entre treino e teste e na implementação dos primeiros modelos de classificação.

As próximas etapas serão:

1. Separar features e labels;
2. Aplicar padronização dos dados;
3. Definir estratégia de divisão entre treino e teste;
4. Implementar um modelo baseline;
5. Comparar modelos mais avançados, como CNN, LSTM e Transformer.